# MediaForge on a free GPU (Google Colab)

Runs the **whole MediaForge app on a free Colab T4 GPU (16 GB)** and gives you a
link to open it in your browser. Motion generation (LTX-Video / Wan) runs on the
GPU for **free** — no API keys, no ngrok account. An optional cell also sets up
**Talking Avatar** (SadTalker).

**Before running: Runtime → Change runtime type → Hardware accelerator → GPU → Save.**

Then run each cell in order with Shift+Enter. The last cell prints a link — click it.

## 1 · Confirm the GPU is on

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Do Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

## 2 · Get MediaForge + install dependencies (~2 min)

In [ ]:
import os
if not os.path.isdir('/content/mediaforge'):
    !git clone -q https://github.com/bigmo6286/mediaforge.git /content/mediaforge
%cd /content/mediaforge/backend
# Torch is already installed on Colab with CUDA; add the rest.
!pip install -q fastapi 'uvicorn[standard]' python-multipart httpx imageio-ffmpeg Pillow
!pip install -q diffusers transformers accelerate sentencepiece
# Local voice (Piper) so the avatar's 'write a script' + Preview voice work.
!pip install -q piper-tts && python -m piper.download_voices en_US-amy-medium en_US-ryan-high --data-dir voices
print('\nInstalled.')

## 3 · (Optional) Save renders to your Google Drive
By default, renders are saved inside this temporary Colab session and vanish
when it ends (you can still download each one from the app). Run this cell to
save them to **Google Drive** instead: `MyDrive/MediaForge/outputs`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MEDIAFORGE_DATA'] = '/content/drive/MyDrive/MediaForge'
os.makedirs(os.environ['MEDIAFORGE_DATA'] + '/outputs', exist_ok=True)
print('Renders will be saved to:', os.environ['MEDIAFORGE_DATA'] + '/outputs')

## 4 · Run everything locally on the GPU (no keys)

In [ ]:
import os
os.environ['MOTION_MODEL'] = 'ltx'        # efficient video model
os.environ['MOTION_PROVIDER'] = 'local'   # run on this GPU
os.environ['WAN_PROVIDER'] = 'local'
print('Configured for local GPU generation.')

## 5 · (Optional) Talking Avatar — set up SadTalker
Makes the **Talking Avatar** tab work on the GPU. SadTalker needs the older
NumPy 1.23 (Colab uses NumPy 2 for Torch/OpenCV), so this builds SadTalker its
own **isolated environment** and points MediaForge at it. First run takes a few
minutes (installs Torch + ~2 GB checkpoints). **Skip if you only want motion.**

In [ ]:
import os
if not os.path.isdir('/content/SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
# SadTalker's stack (torch 2.0.1 / numpy 1.23) only has wheels for Python
# 3.10, not Colab's 3.12. Use `uv` to build an isolated Python 3.10 env.
!pip install -q uv
!rm -rf /content/st-venv
!uv venv --python 3.10 /content/st-venv
VP = '/content/st-venv/bin/python'
# Torch 2.0.1 + torchvision 0.15 (has functional_tensor) on CUDA 11.8.
!uv pip install --python $VP torch==2.0.1 torchvision==0.15.2 \
    --index-url https://download.pytorch.org/whl/cu118
# SadTalker deps (numpy-1.23 era).
!uv pip install --python $VP numpy==1.23.5 scipy==1.10.1 numba resampy \
    librosa==0.9.2 pydub yacs safetensors tqdm pyyaml joblib scikit-image imageio \
    imageio-ffmpeg gfpgan==1.3.8 basicsr==1.4.2 facexlib==0.3.0 kornia==0.6.8 \
    face-alignment==1.3.5
# Sanity-check the isolated env actually has torch+cuda.
!$VP -c "import torch, numpy; print('venv torch', torch.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"
# Download the SadTalker checkpoints (~2 GB).
!cd /content/SadTalker && bash scripts/download_models.sh
os.environ['SADTALKER_DIR'] = '/content/SadTalker'
os.environ['SADTALKER_PYTHON'] = VP
os.environ['AVATAR_PROVIDER'] = 'local'
print('\nSadTalker ready (isolated Python 3.10 env) — Talking Avatar runs on this GPU.')

## 5b · (Optional) Quick avatar smoke-test
Renders a short clip from a bundled example face + a Piper voice line to confirm
SadTalker works, and shows it inline. Takes ~1-3 min. Run only after cell 5.

In [ ]:
import subprocess, glob, os
from IPython.display import Video
# short test voice via Piper (installed in cell 2, Colab's main env)
voices = glob.glob('/content/mediaforge/backend/voices/*.onnx')
assert voices, 'No Piper voice found - rerun cell 2.'
subprocess.run(['piper','-m',voices[0],'-f','/content/test_audio.wav'],
               input='Hi, this is a quick MediaForge avatar test.', text=True)
# run SadTalker via its ISOLATED python on a bundled example image
img = sorted(glob.glob('/content/SadTalker/examples/source_image/*.png'))[0]
print('source image:', img, '\nrendering (this takes a minute or two)...')
r = subprocess.run(['/content/st-venv/bin/python','inference.py',
    '--source_image',img,'--driven_audio','/content/test_audio.wav',
    '--result_dir','/content/st_test','--still','--preprocess','full'],
    cwd='/content/SadTalker', capture_output=True, text=True)
mp4 = sorted(glob.glob('/content/st_test/**/*.mp4', recursive=True), key=os.path.getmtime)
if mp4:
    print('\nSUCCESS - SadTalker works. Rendered:', mp4[-1])
else:
    print('\nFAILED - last output:\n', r.stdout[-1500:], '\n', r.stderr[-1800:])
Video(mp4[-1], embed=True, width=320) if mp4 else None

## 6 · Start MediaForge and open it
Run this, wait ~10 seconds, then click the printed link.

In [ ]:
import subprocess, time, sys, os
# Start the server in the background (serves the UI + API on port 8000).
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app',
                         '--host', '0.0.0.0', '--port', '8000'],
                        cwd='/content/mediaforge/backend', env=os.environ.copy())
time.sleep(10)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('\n==============================================')
print(' Open MediaForge here:')
print(' ', url)
print('==============================================')
print('\nMotion tab: type a prompt and Generate.')
print('Talking Avatar tab (if you ran cell 5): upload a face + script, Create.')
print('First run of each downloads weights (a few minutes). Keep this tab open.')

## Where do my files go?
* Easiest: the **download button** under each render in the Results panel.
* On the server: `data/outputs/` (this Colab session, or your Google Drive if
  you ran cell 3).

## Notes
* Free Colab sessions time out after a while — rerun the cells for a fresh one.
* SadTalker is heavier/pickier than motion; if avatars fail, restart the runtime
  and run the cells again, or use a small hosted fal credit instead.
* To stop: Runtime → Disconnect and delete runtime.